In [15]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (ConfusionMatrixDisplay, classification_report,
                             precision_recall_fscore_support, roc_auc_score, roc_curve)
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [16]:
RANDOM_STATE = 42
os.makedirs("reports/figures", exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")
 
df = pd.read_csv("D:/My Resume/IBM_TELECOM/telcom/telco_clean.csv")

In [17]:
# 1. Choose features (and avoid data leakage)
# ---------------------------------------------------------------
# Leakage = columns that reveal the answer or only exist after the customer left:
#   churn_label (same as target), churn_reason (only filled for churners),
#   churn_score (IBM's own prediction). Excluded on purpose.
# Also excluded: ids, location columns (too many unique values), and cltv
# (a derived value estimate; we use it for the revenue analysis instead).
TARGET = "churn_value"
DROP = ["customer_id", "churn_label", "churn_reason", "churn_score", "cltv",
        "city", "zip_code", "lat_long", "latitude", "longitude", "tenure_bucket", "tenurebucket"]
X = df.drop(columns=[TARGET] + [c for c in DROP if c in df.columns])
y = df[TARGET]
 
numeric_cols = X.select_dtypes(include="number").columns.tolist()
categorical_cols = X.select_dtypes(exclude="number").columns.tolist()
print(f"Features: {len(numeric_cols)} numeric, {len(categorical_cols)} categorical")
 
preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
])

Features: 4 numeric, 16 categorical


In [18]:
# 2. Train / test split (stratified keeps the ~27% churn ratio in both sets)
# ---------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)
print(f"Train: {len(X_train)} rows | Test: {len(X_test)} rows")
 
# class_weight="balanced" makes the model pay more attention to the churners,
# which fixes the imbalance problem (accuracy alone would be misleading).
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced_subsample",
        n_jobs=-1, random_state=RANDOM_STATE),
}


Train: 5634 rows | Test: 1409 rows


In [19]:
# 3. Fit, evaluate, compare
# ---------------------------------------------------------------
fitted, results = {}, []
for name, clf in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", clf)])
    pipe.fit(X_train, y_train)
    proba = pipe.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(y_test, pred, average="binary")
    results.append({"model": name, "precision": p, "recall": r, "f1": f1,
                    "roc_auc": roc_auc_score(y_test, proba),
                    "accuracy": (pred == y_test).mean()})
    fitted[name] = (pipe, proba)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, pred, target_names=["Stayed", "Churned"]))
 
# Baseline: always predicting "stayed" gets ~73% accuracy but 0% recall.
baseline_acc = 1 - y_test.mean()
res = pd.DataFrame(results).set_index("model").round(3)
print("\nModel comparison (test set):\n", res)
print(f"Baseline 'nobody churns' accuracy: {baseline_acc:.3f} (recall = 0)")
 
# Cross-validation on the best model, to show the result is stable
best_name = res["roc_auc"].idxmax()
best_pipe = Pipeline([("prep", preprocess), ("model", models[best_name])])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
oof = cross_val_predict(best_pipe, X, y, cv=skf, method="predict_proba")[:, 1]
print(f"\nBest model by ROC-AUC: {best_name} (5-fold out-of-fold AUC = {roc_auc_score(y, oof):.3f})")


=== Logistic Regression ===
              precision    recall  f1-score   support

      Stayed       0.90      0.73      0.81      1035
     Churned       0.51      0.78      0.62       374

    accuracy                           0.74      1409
   macro avg       0.71      0.76      0.71      1409
weighted avg       0.80      0.74      0.76      1409


=== Random Forest ===
              precision    recall  f1-score   support

      Stayed       0.89      0.79      0.84      1035
     Churned       0.56      0.73      0.63       374

    accuracy                           0.78      1409
   macro avg       0.72      0.76      0.74      1409
weighted avg       0.80      0.78      0.78      1409


Model comparison (test set):
                      precision  recall     f1  roc_auc  accuracy
model                                                           
Logistic Regression      0.511   0.781  0.618    0.849     0.744
Random Forest            0.558   0.730  0.633    0.853     0.775
Bas

In [20]:
# 4. Charts: ROC curves and confusion matrix
# ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
for name, (_, proba) in fitted.items():
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, lw=2.5, label=f"{name} (AUC {roc_auc_score(y_test, proba):.2f})")
ax.plot([0, 1], [0, 1], "k--", lw=1)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate (recall)")
ax.set_title("ROC curves", loc="left", fontweight="bold")
ax.legend(fontsize=11)
plt.tight_layout()
fig.savefig("reports/figures/09_roc_curves.png", dpi=150)
plt.close(fig)
 
fig, ax = plt.subplots(figsize=(6, 5))
best_test_pred = (fitted[best_name][1] >= 0.5).astype(int)
ConfusionMatrixDisplay.from_predictions(
    y_test, best_test_pred, display_labels=["Stayed", "Churned"], cmap="Reds", ax=ax, colorbar=False)
ax.set_title(f"Confusion matrix: {best_name}", loc="left", fontweight="bold", fontsize=14)
plt.tight_layout()
fig.savefig("reports/figures/10_confusion_matrix.png", dpi=150)
plt.close(fig)

In [21]:
# 5. What drives churn? Logistic regression coefficients (easy to explain)
# ---------------------------------------------------------------
lr_pipe = fitted["Logistic Regression"][0]
names = lr_pipe.named_steps["prep"].get_feature_names_out()
names = [n.replace("num__", "").replace("cat__", "") for n in names]
coefs = pd.Series(lr_pipe.named_steps["model"].coef_[0], index=names).sort_values()
top = pd.concat([coefs.head(8), coefs.tail(8)])
fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top.index, top.values, color=["#4c8c4a" if v < 0 else "#d1495b" for v in top.values])
ax.set_title("What drives churn (logistic regression)", loc="left", fontweight="bold")
ax.set_xlabel("Coefficient  (red = more churn, green = less churn)")
plt.tight_layout()
fig.savefig("reports/figures/11_feature_drivers.png", dpi=150)
plt.close(fig)
print("\nTop churn-increasing features:\n", coefs.tail(6).round(2).to_string())
print("\nTop churn-reducing features:\n", coefs.head(6).round(2).to_string())
 


Top churn-increasing features:
 streaming_movies_Yes               0.08
payment_method_Electronic check    0.25
total_charges                      0.51
contract_Month-to-month            0.67
dependents_No                      0.73
internet_service_Fiber optic       0.85

Top churn-reducing features:
 tenure_months         -1.21
internet_service_No   -0.99
dependents_Yes        -0.91
contract_Two year     -0.74
streaming_movies_No   -0.27
streaming_tv_No       -0.26


In [22]:
# 6. Pick an action threshold (precision vs recall trade-off)
# ---------------------------------------------------------------
proba_best = fitted[best_name][1]
rows = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (proba_best >= t).astype(int)
    p, r, _, _ = precision_recall_fscore_support(y_test, pred, average="binary", zero_division=0)
    rows.append({"threshold": t, "flagged": int(pred.sum()), "precision": p, "recall": r})
print("\nThreshold trade-off (test set):\n", pd.DataFrame(rows).round(2).to_string(index=False))


Threshold trade-off (test set):
  threshold  flagged  precision  recall
       0.3      718       0.47    0.90
       0.4      600       0.52    0.83
       0.5      489       0.56    0.73
       0.6      387       0.61    0.63
       0.7      270       0.67    0.48


In [23]:
# 7. Score every customer (out-of-fold, so nobody is scored by a model that saw them)
# ---------------------------------------------------------------
scores = df[["customer_id", "contract", "tenure_months", "internet_service",
             "monthly_charges", TARGET]].copy()
scores["churn_probability"] = oof.round(3)
scores["risk_tier"] = pd.cut(scores["churn_probability"], bins=[-0.01, 0.4, 0.7, 1.0],
                             labels=["Low", "Medium", "High"])
scores.to_csv("D:/My Resume/IBM_TELECOM/telcom/customer_risk_scores.csv", index=False)

In [24]:
# 8. Business impact (EDIT THE ASSUMPTIONS to whatever you can justify)
# ---------------------------------------------------------------
SAVE_RATE = 0.20        # assumption: share of flagged churners a retention offer keeps
OFFER_COST = 30         # assumption: cost per customer contacted ($)
high = scores[scores["risk_tier"] == "High"]
real_churners_high = high[high[TARGET] == 1]
annual_at_risk = real_churners_high["monthly_charges"].sum() * 12
saved = annual_at_risk * SAVE_RATE
cost = len(high) * OFFER_COST
print("\n--- Business impact (out-of-fold, all customers) ---")
print(f"High-risk customers flagged: {len(high):,} "
      f"({(high[TARGET] == 1).mean():.1%} of them actually churned)")
print(f"Share of all churners caught in High tier: {len(real_churners_high) / y.sum():.1%}")
print(f"Annual revenue held by those churners: ${annual_at_risk:,.0f}")
print(f"If {SAVE_RATE:.0%} are retained: +${saved:,.0f}/yr, campaign cost ${cost:,.0f}, "
      f"net ${saved - cost:,.0f}")
print("Saved: data/customer_risk_scores.csv and 3 charts in reports/figures/")


--- Business impact (out-of-fold, all customers) ---
High-risk customers flagged: 1,303 (68.4% of them actually churned)
Share of all churners caught in High tier: 47.7%
Annual revenue held by those churners: $853,478
If 20% are retained: +$170,696/yr, campaign cost $39,090, net $131,606
Saved: data/customer_risk_scores.csv and 3 charts in reports/figures/


In [25]:
print(df[df.churn_value==1]["churn_reason"].value_counts().head(5))
print(df[["churn_value","tenure_months","monthly_charges"]].corr().round(2))

churn_reason
Attitude of support person                   192
Competitor offered higher download speeds    189
Competitor offered more data                 162
Don't know                                   154
Competitor made better offer                 140
Name: count, dtype: int64
                 churn_value  tenure_months  monthly_charges
churn_value             1.00          -0.35             0.19
tenure_months          -0.35           1.00             0.25
monthly_charges         0.19           0.25             1.00
